In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from roboflow import Roboflow
from getpass import getpass

api_key = getpass("Roboflow API key paste karo: ")
rf = Roboflow(api_key=api_key)

dataset = (rf.workspace("kushal-workspace")
             .project("employee-activity-detection")
             .version(7)
             .download("yolov8"))
print("Dataset yahan hai:", dataset.location)

Roboflow API key paste karo: ··········
loading Roboflow workspace...
loading Roboflow project...
Dataset yahan hai: /content/Employee-Activity-Detection-7


In [ ]:


!ls {dataset.location}
!cat {dataset.location}/data.yaml

data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid
names:
- Notworking
- Working
nc: 2
path: /content/Employee-Activity-Detection-7
roboflow:
  license: CC BY 4.0
  project: employee-activity-detection
  url: https://universe.roboflow.com/kushal-workspace/employee-activity-detection/dataset/7
  version: 7
  workspace: kushal-workspace
test: test/images
train: train/images
val: valid/images


In [ ]:


import glob, collections, yaml

root = dataset.location
names = yaml.safe_load(open(f"{root}/data.yaml"))["names"]

for split in ["train", "valid", "test"]:
    imgs = len(glob.glob(f"{root}/{split}/images/*"))
    c = collections.Counter()
    for f in glob.glob(f"{root}/{split}/labels/*.txt"):
        for line in open(f):
            c[names[int(line.split()[0])]] += 1
    print(split, "images:", imgs, "boxes:", dict(c))

train images: 1178 boxes: {'Notworking': 2017, 'Working': 6473}
valid images: 333 boxes: {'Notworking': 580, 'Working': 1761}
test images: 177 boxes: {'Working': 1018, 'Notworking': 304}


In [ ]:


import yaml

p = f"{dataset.location}/data.yaml"
cfg = yaml.safe_load(open(p))
cfg["path"] = dataset.location
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg["test"] = "test/images"
yaml.safe_dump(cfg, open(p, "w"))
print("data.yaml theek kar di")

data.yaml theek kar di


In [ ]:

from ultralytics import YOLO

DATA = f"{dataset.location}/data.yaml"
RUNS = "/content/drive/MyDrive/yolo_runs"
model = YOLO("yolo26n.pt")
_ = model.train(
    data=DATA,
    patience=10,
    imgsz=640,
    freeze=10,
    project=RUNS,
    name="stage1",
    exist_ok=True,
)

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Employee-Activity-Detection-7/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=stage1, nbs=64, nms=None, opset=None, optim

In [ ]:

from ultralytics import YOLO

DATA = f"{dataset.location}/data.yaml"
RUNS = "/content/drive/MyDrive/yolo_runs"

model = YOLO(f"{RUNS}/stage1/weights/best.pt")
_ = model.train(
    data=DATA,
    epochs=100,
    patience=10,
    imgsz=640,
    optimizer="AdamW",
    lr0=0.001,
    project=RUNS,
    name="stage2",
    exist_ok=True,
)

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Employee-Activity-Detection-7/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/yolo_runs/stage1/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

In [ ]:

import pandas as pd

RUNS = "/content/drive/MyDrive/yolo_runs"
for s in ["stage1", "stage2"]:
    df = pd.read_csv(f"{RUNS}/{s}/results.csv")
    df.columns = df.columns.str.strip()
    i = df["metrics/mAP50-95(B)"].idxmax()
    print(s,
          "| best epoch:", int(df.loc[i, "epoch"]),
          "| mAP50-95:", round(df.loc[i, "metrics/mAP50-95(B)"], 3),
          "| mAP50:", round(df.loc[i, "metrics/mAP50(B)"], 3),
          "| epochs chale:", len(df))

stage1 | best epoch: 47 | mAP50-95: 0.675 | mAP50: 0.962 | epochs chale: 57
stage2 | best epoch: 31 | mAP50-95: 0.695 | mAP50: 0.969 | epochs chale: 41


test

In [ ]:

!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics roboflow

Sun Oct  4 09:41:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:


!ls -lh /content/drive/MyDrive/yolo_runs/stage1/weights/content/drive/MyDrive/yolo_runs/stage2/weights

/content/drive/MyDrive/yolo_runs/stage1/weights:
total 11M
-rw------- 1 root root 5.2M Oct  3 16:29 best.pt
-rw------- 1 root root 5.2M Oct  3 16:29 last.pt

/content/drive/MyDrive/yolo_runs/stage2/weights:
total 11M
-rw------- 1 root root 5.2M Oct  3 16:45 best.pt
-rw------- 1 root root 5.2M Oct  3 16:45 last.pt


In [ ]:

from roboflow import Roboflow
from getpass import getpass
import yaml

rf = Roboflow(api_key=getpass("Roboflow API key paste karo: "))
dataset = (rf.workspace("kushal-workspace")
             .project("employee-activity-detection")
             .version(7)
             .download("yolov8"))

p = f"{dataset.location}/data.yaml"
cfg = yaml.safe_load(open(p))
cfg["path"] = dataset.location
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg["test"] = "test/images"
yaml.safe_dump(cfg, open(p, "w"))
print("Dataset tayyar:", dataset.location)

Roboflow API key paste karo: ··········
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Employee-Activity-Detection-7 in yolov8:: 100%|██████████| 3388/3388 [00:00<00:00, 8252.05it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Dataset tayyar: /content/Employee-Activity-Detection-7


In [ ]:
from ultralytics import YOLO

DATA = f"{dataset.location}/data.yaml"
RUNS = "/content/drive/MyDrive/yolo_runs"

best = YOLO(f"{RUNS}/stage2/weights/best.pt")
_ = best.val(data=DATA, split="test", project=RUNS, name="test", exist_ok=True)

Ultralytics 8.4.172 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 120 layers, 2,375,226 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1531.2±378.7 MB/s, size: 46.7 KB)
val: Scanning /content/Employee-Activity-Detection-7/test/labels... 177 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 177/177 1.4Kit/s 0.1s
val: New cache created: /content/Employee-Activity-Detection-7/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 3.5it/s 3.4s
                   all        177       1322      0.956      0.949      0.972      0.674
            Notworking        148        304      0.941      0.918      0.964       0.65
               Working        172       1018      0.971      0.979       0.98      0.699
Speed: 1.6ms preprocess, 4.0ms inference, 0.0ms loss, 3.5ms postprocess per image
Results saved to /content/drive/MyDrive/yolo_r

In [ ]:
import shutil, os

os.makedirs("/content/drive/MyDrive/final_model", exist_ok=True)
dst = "/content/drive/MyDrive/final_model/office_activity_yolo26n.pt"
shutil.copy("/content/drive/MyDrive/yolo_runs/stage2/weights/best.pt", dst)
print("Saved:", dst)

Saved: /content/drive/MyDrive/final_model/office_activity_yolo26n.pt


demo


In [ ]:

!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

/bin/bash: line 1: nvidia-smi: command not found
Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 5.7 MB/s eta 0:00:00


In [ ]:
import glob

for f in glob.glob("/content/drive/MyDrive/**/*.mp4", recursive=True):
    print(f)
!ls -lh /content/drive/MyDrive/yolo_runs/stage2/weights/best.pt

/content/drive/MyDrive/projectvideo.mp4
/content/drive/MyDrive/video2.mp4
/content/drive/MyDrive/yolo_runs/demo/demo_final.mp4
/content/drive/MyDrive/yolo_runs/demo3/demo_final.mp4
/content/drive/MyDrive/yolo_runs/sweep_c25_s640/final.mp4
/content/drive/MyDrive/yolo_runs/sweep_c50_s640/final.mp4
/content/drive/MyDrive/yolo_runs/sweep_c25_s960/final.mp4
/content/drive/MyDrive/yolo_runs/sweep_c50_s960/final.mp4
-rw------- 1 root root 5.2M Oct  3 16:45 /content/drive/MyDrive/yolo_runs/stage2/weights/best.pt


In [ ]:
import cv2, os, subprocess
from collections import deque
from ultralytics import YOLO

VIDEO = "/content/drive/MyDrive/yolo_runs/video5.mp4"
NAME = "video5newcode"
CONF = 0.4
PERSON_CONF = 0.3
SMOOTH_FRAMES = 15
KEEP_FRAMES = 10
MAX_SECONDS = 10

RUNS = "/content/drive/MyDrive/yolo_runs"
OUT_DIR = f"{RUNS}/filtered"
os.makedirs(OUT_DIR, exist_ok=True)

person_model = YOLO("yolo26n.pt")
our_model = YOLO(f"{RUNS}/stage2/weights/best.pt")

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union if union > 0 else 0

cap = cv2.VideoCapture(VIDEO)
if not cap.isOpened():
    raise SystemExit(f"Video nahi mili ya khul nahi saki: {VIDEO}")
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25
max_frames = int(fps * MAX_SECONDS) if MAX_SECONDS else 10**9
raw = f"/content/{NAME}_raw.mp4"
final = f"{OUT_DIR}/{NAME}.mp4"
writer = cv2.VideoWriter(raw, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

state = {}

n = 0
while n < max_frames:
    ok, frame = cap.read()
    if not ok:
        break
    n += 1

    res = person_model.track(frame, persist=True, classes=[0], conf=PERSON_CONF,
                             verbose=False)[0]
    if res.boxes.id is not None:
        pboxes = res.boxes.xyxy.cpu().numpy()
        pids = res.boxes.id.int().cpu().tolist()
    else:
        pboxes, pids = [], []

    ours = our_model.predict(frame, conf=CONF, imgsz=640, verbose=False)[0]
    assigned = {}
    for box, cls, cf in zip(ours.boxes.xyxy.cpu().numpy(),
                            ours.boxes.cls.cpu().numpy(),
                            ours.boxes.conf.cpu().numpy()):
        cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
        inside = [k for k, p in enumerate(pboxes)
                  if p[0] <= cx <= p[2] and p[1] <= cy <= p[3]]
        if not inside:
            continue
        k = max(inside, key=lambda j: iou(box, pboxes[j]))
        assigned[k] = (float(cf), int(cls))

    seen = set()
    for k, tid in enumerate(pids):
        st = state.setdefault(tid, {"box": None, "votes": deque(maxlen=SMOOTH_FRAMES), "missed": 0})
        st["box"], st["missed"] = pboxes[k], 0
        seen.add(tid)
        if k in assigned:
            st["votes"].append(assigned[k])
    for tid, st in state.items():
        if tid not in seen:
            st["missed"] += 1
    for tid, st in state.items():
        if st["missed"] > KEEP_FRAMES or not st["votes"]:
            continue
        score = {}
        for cf, cls in st["votes"]:
            score[cls] = score.get(cls, 0) + cf
        cls = max(score, key=score.get)
        n_cls = sum(1 for _, c in st["votes"] if c == cls)
        cf_avg = score[cls] / n_cls
        label = our_model.names[cls]
        color = (0, 200, 0) if label == "Working" else (0, 0, 255)
        x1, y1, x2, y2 = (int(v) for v in st["box"])
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{label} {cf_avg:.2f}", (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    writer.write(frame)

cap.release()
writer.release()
subprocess.run(["ffmpeg", "-y", "-i", raw, "-vcodec", "libx264",
                "-pix_fmt", "yuv420p", final], check=True)
print("Tayyar:", final)

Tayyar: /content/drive/MyDrive/yolo_runs/filtered/video5newcode.mp4
